# Module 2 · Chapter 1 — 확률이란 무엇인가

본문(.md)과 짝이 되는 실습 노트북입니다.
- **본문**: 확률의 세 해석, Kolmogorov 공리, 덧셈 법칙
- **이 노트북**: 시뮬레이션으로 확률의 수렴을 눈으로 확인하고, 덧셈 법칙을 검증

## 0. 환경 준비

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(seed=42)
sns.set_theme(context="notebook", style="whitegrid")

## 1. 빈도론적 수렴 — 동전을 많이 던질수록 0.5에 가까워진다

"같은 실험을 무한 반복하면 몇 번 발생하는가?"가 빈도론적 확률의 정의입니다.
동전 던지기 시뮬레이션으로 이것이 실제로 수렴하는지 봅니다.

In [ ]:
# 동전 10000번 던지기 (1=앞면, 0=뒷면)
flips = rng.integers(0, 2, size=10_000)

# 누적 앞면 비율
cumulative_prob = np.cumsum(flips) / np.arange(1, len(flips) + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(cumulative_prob, linewidth=1, color="steelblue", label="누적 앞면 비율")
ax.axhline(0.5, color="tomato", linestyle="--", linewidth=2, label="이론적 확률 0.5")
ax.set_xscale("log")
ax.set_xlabel("던진 횟수 (로그 스케일)")
ax.set_ylabel("누적 앞면 비율")
ax.set_title("대수의 법칙: 시행 수가 늘수록 빈도가 확률에 수렴한다")
ax.set_ylim(0.3, 0.7)
ax.legend()
plt.tight_layout()
plt.show()

print(f"최종 앞면 비율: {cumulative_prob[-1]:.4f} (이론값 0.5000)")

## 2. 시나리오 — 독감 자가진단 키트

1000명을 시뮬레이션합니다.
- 독감 감염률: 5% (겨울철 피크 시즌)
- 키트 민감도(sensitivity): 90%
- 키트 특이도(specificity): 95%

In [ ]:
n_people = 10_000
prevalence = 0.05   # 독감 감염률
sensitivity = 0.90  # 민감도
specificity = 0.95  # 특이도

# 감염 여부
infected = rng.random(n_people) < prevalence

# 검사 결과
test_pos = np.where(
    infected,
    rng.random(n_people) < sensitivity,    # 감염자: 민감도 확률로 양성
    rng.random(n_people) > specificity     # 정상인: (1-특이도) 확률로 위양성
)

# 2×2 혼동행렬
tp = np.sum(infected & test_pos)
fp = np.sum(~infected & test_pos)
fn = np.sum(infected & ~test_pos)
tn = np.sum(~infected & ~test_pos)

cm = pd.DataFrame(
    [[tp, fn], [fp, tn]],
    index=["감염 (실제)", "정상 (실제)"],
    columns=["양성 (검사)", "음성 (검사)"]
)
print(cm)
print(f"\n전체 양성: {tp+fp}명")
print(f"양성 중 실제 감염: {tp}명")
print(f"양성 예측도(PPV) = {tp/(tp+fp):.3f}")
print(f"\n※ 민감도 90%에도 불구하고, 양성이라도 실제 감염 확률은 약 {tp/(tp+fp)*100:.0f}%입니다")

## 3. Kolmogorov 공리 검증

덧셈 법칙: $P(A \cup B) = P(A) + P(B) - P(A \cap B)$

두 사건의 합집합 확률을 두 가지 방법으로 계산해 같은지 확인합니다.

In [ ]:
# A: 양성 판정 (test_pos), B: 실제 감염 (infected)
p_a = test_pos.mean()           # P(양성)
p_b = infected.mean()           # P(감염)
p_a_and_b = (test_pos & infected).mean()  # P(양성 AND 감염)
p_a_or_b  = (test_pos | infected).mean()  # P(양성 OR 감염) — 직접 계산

p_a_or_b_formula = p_a + p_b - p_a_and_b  # 덧셈 법칙

print(f"P(양성)          = {p_a:.4f}")
print(f"P(감염)          = {p_b:.4f}")
print(f"P(양성 AND 감염) = {p_a_and_b:.4f}")
print()
print(f"직접 계산 P(A∪B) = {p_a_or_b:.4f}")
print(f"덧셈 법칙으로   = {p_a_or_b_formula:.4f}")
print(f"\n두 값이 같습니까? {abs(p_a_or_b - p_a_or_b_formula) < 1e-10}")

## 4. 감염률이 다를 때 PPV가 어떻게 변하는가

같은 키트(민감도·특이도 동일)라도 *감염률* 이 달라지면 양성 예측도가 크게 달라집니다.

In [ ]:
prevalences = np.linspace(0.001, 0.30, 100)

def ppv(prev, sens, spec):
    tp = sens * prev
    fp = (1 - spec) * (1 - prev)
    return tp / (tp + fp)

ppv_values = [ppv(p, sensitivity, specificity) for p in prevalences]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(prevalences * 100, [v * 100 for v in ppv_values],
        color="steelblue", linewidth=2)
ax.axvline(prevalence * 100, color="tomato", linestyle="--",
           label=f"현재 감염률 {prevalence*100:.0f}%")
ax.axhline(ppv(prevalence, sensitivity, specificity) * 100,
           color="tomato", linestyle=":", alpha=0.7)
ax.set_xlabel("감염률 (%)")
ax.set_ylabel("양성 예측도 PPV (%)")
ax.set_title("감염률에 따른 양성 예측도 변화\n(민감도 90%, 특이도 95% 고정)")
ax.legend()
plt.tight_layout()
plt.show()

print("→ 감염률이 낮은 시즌(봄·여름)에 양성이어도 실제 독감 가능성은 더 낮습니다.")
print("  기저율(base rate)이 확률 해석의 핵심입니다.")

## 5. 직접 답해 보기

1. '빈도론적 확률'과 '주관적 확률'의 차이를 자가진단 키트 시나리오로 설명한다면?
2. 위 그래프에서 PPV가 감염률에 비선형적으로 반응하는 이유는 무엇일까?
3. Kolmogorov 공리 중 "여사건 확률" 규칙을 이 시나리오에서 어떻게 확인할 수 있을까?